In [21]:
from pathlib import Path

import os



import pandas as pd

from datasets import load_dataset

from dotenv import load_dotenv



load_dotenv()



local_dataset_path = os.getenv("LOCAL_DATASET_PATH", "").strip()

hf_dataset_id = os.getenv(

    "HF_DATASET_ID", "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

).strip()

hf_dataset_split = os.getenv("HF_DATASET_SPLIT", "train").strip()



if local_dataset_path:

    path = Path(local_dataset_path)

    if not path.exists():

        raise FileNotFoundError(f"Dataset not found: {path}")



    suffix = path.suffix.lower()

    if suffix == ".csv":

        df = pd.read_csv(path)

    elif suffix in {".json", ".jsonl"}:

        df = pd.read_json(path, lines=(suffix == ".jsonl"))

    elif suffix == ".parquet":

        df = pd.read_parquet(path)

    else:

        raise ValueError("Unsupported file format. Use csv, json, jsonl, or parquet.")



    print(f"Loaded local dataset: {path}")

    print(f"Shape: {df.shape}")

    display(df.head())

else:

    ds = load_dataset(hf_dataset_id, split=hf_dataset_split)

    print(f"Loaded Hugging Face dataset: {hf_dataset_id} [{hf_dataset_split}]")

    print(ds)

    display(ds.to_pandas().head())


Loaded Hugging Face dataset: bitext/Bitext-customer-support-llm-chatbot-training-dataset [train]
Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [22]:
import shutil

from pathlib import Path



output_dir = Path("data/full_dataset")

output_dir.mkdir(parents=True, exist_ok=True)



if "ds" in globals() and local_dataset_path == "":

    hf_dir = output_dir / "hf_dataset"

    if hf_dir.exists():

        shutil.rmtree(hf_dir)



    ds.save_to_disk(str(hf_dir))

    csv_path = output_dir / "dataset.csv"

    ds.to_csv(str(csv_path))



    print(f"Saved Hugging Face dataset folder to: {hf_dir}")

    print(f"Saved CSV export to: {csv_path}")

    print(f"Rows saved: {len(ds)}")

elif "df" in globals():

    csv_path = output_dir / "dataset.csv"

    df.to_csv(csv_path, index=False)



    print(f"Saved dataset CSV to: {csv_path}")

    print(f"Rows saved: {len(df)}")

else:

    raise ValueError("No dataset found. Run Cell 1 first to load the dataset.")


Creating CSV from Arrow format: 100%|██████████| 27/27 [00:00<00:00, 74.86ba/s]

Saved Hugging Face dataset folder to: data/full_dataset/hf_dataset
Saved CSV export to: data/full_dataset/dataset.csv
Rows saved: 26872


In [23]:
import json

from pathlib import Path



from jinja2 import Template

from openai import OpenAI



api_key = os.getenv("OPENAI_API_KEY", "").strip()

model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip()



if not api_key:

    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")



extract_prompt_path = Path("support_ticket_triage_prompt.j2")

review_prompt_path = Path("support_ticket_triage_review_prompt.j2")



if not extract_prompt_path.exists():

    raise FileNotFoundError("support_ticket_triage_prompt.j2 not found in project root.")

if not review_prompt_path.exists():

    raise FileNotFoundError("support_ticket_triage_review_prompt.j2 not found in project root.")



extract_template = Template(extract_prompt_path.read_text(encoding="utf-8"))

review_template = Template(review_prompt_path.read_text(encoding="utf-8"))

client = OpenAI(api_key=api_key)



sample_df = df.head(5).copy() if "df" in globals() else ds.to_pandas().head(5).copy()



def build_request_text(row_payload: dict) -> str:

    for key in ("instruction", "text", "utterance"):

        if key in row_payload:

            return str(row_payload[key])

    return json.dumps(row_payload, ensure_ascii=False)



def call_llm_json(user_prompt: str) -> tuple[dict, str]:

    response = client.chat.completions.create(

        model=model_name,

        messages=[

            {"role": "system", "content": "Return valid JSON only."},

            {"role": "user", "content": user_prompt},

        ],

        temperature=0,

    )

    raw_content = (response.choices[0].message.content or "").strip()

    try:

        return json.loads(raw_content), raw_content

    except json.JSONDecodeError:

        return {"raw_output": raw_content}, raw_content



results = []

for row_index, row in sample_df.iterrows():

    row_payload = {key: value for key, value in row.to_dict().items() if pd.notna(value)}

    request_text = build_request_text(row_payload)



    extract_prompt = extract_template.render(request=request_text)

    call1_parsed, _ = call_llm_json(extract_prompt)



    extraction_for_review = {

        "intent": call1_parsed.get("intent"),

        "symptoms": call1_parsed.get("symptoms", []),

    }

    review_prompt = review_template.render(

        request=request_text,

        extraction=json.dumps(extraction_for_review, ensure_ascii=False),

    )

    call2_parsed, _ = call_llm_json(review_prompt)



    results.append(

        {

            "row_index": int(row_index),

            "request": request_text,

            "call1_intent": extraction_for_review.get("intent"),

            "call1_symptoms": extraction_for_review.get("symptoms", []),

            "review": call2_parsed.get("review"),

            "final_intent": call2_parsed.get("corrected_intent"),

            "final_symptoms": call2_parsed.get("corrected_symptoms", []),

        }

    )



results_df = pd.DataFrame(results)

display(results_df)


,row_index,request,call1_intent,call1_symptoms,review,final_intent,final_symptoms
0,0,question about cancelling order {{Order Number}},cancel order,[question about cancelling order],The intent 'cancel order' is accurate and spec...,cancel order,"[question, cancelling order, Order Number]"
1,1,i have a question about cancelling oorder {{Or...,cancel order,"[question about cancelling an order, specific ...",The intent 'cancel order' is accurate but coul...,inquire about cancelling order with specific o...,"[question about cancelling an order, specific ..."
2,2,i need help cancelling puchase {{Order Number}},cancel purchase,"[request to cancel order, providing order number]",The intent is accurate and specific enough as ...,request help to cancel purchase,"[request for help, cancelling purchase, order ..."
3,3,I need to cancel purchase {{Order Number}},cancel purchase,[],The intent is accurate and specific enough as ...,cancel purchase,[]
4,4,"I cannot afford this order, cancel purchase {{...",cancel purchase,[cannot afford the order],The intent 'cancel purchase' is accurate and s...,cancel purchase,"[cannot afford the order, order number: {{Orde..."


In [24]:
judge_prompt_path = Path("support_ticket_compare_judge_prompt.j2")

if not judge_prompt_path.exists():

    raise FileNotFoundError("support_ticket_compare_judge_prompt.j2 not found in project root.")



if "results_df" not in globals() or results_df.empty:

    raise ValueError("results_df is missing or empty. Run Cell 3 first.")



judge_template = Template(judge_prompt_path.read_text(encoding="utf-8"))



judge_rows = []

for _, row in results_df.iterrows():

    extraction_a = {

        "intent": row.get("call1_intent"),

        "symptoms": row.get("call1_symptoms", []),

    }

    extraction_b = {

        "intent": row.get("final_intent"),

        "symptoms": row.get("final_symptoms", []),

    }



    judge_prompt = judge_template.render(

        request=row.get("request", ""),

        extraction_a=json.dumps(extraction_a, ensure_ascii=False),

        extraction_b=json.dumps(extraction_b, ensure_ascii=False),

    )



    judge_parsed, _ = call_llm_json(judge_prompt)



    judge_rows.append(

        {

            "row_index": row.get("row_index"),

            "winner": judge_parsed.get("winner"),

            "score_a": judge_parsed.get("score_a"),

            "score_b": judge_parsed.get("score_b"),

            "analysis_a": judge_parsed.get("analysis_a"),

            "analysis_b": judge_parsed.get("analysis_b"),

            "judge_final_intent": judge_parsed.get("final_intent"),

            "judge_final_symptoms": judge_parsed.get("final_symptoms", []),

            "judge_notes": judge_parsed.get("notes"),

        }

    )



judge_results_df = pd.DataFrame(judge_rows)

display(judge_results_df)


,row_index,winner,score_a,score_b,analysis_a,analysis_b,judge_final_intent,judge_final_symptoms,judge_notes
0,0,B,6,8,Extraction A accurately captures the intent to...,Extraction B correctly identifies the intent a...,cancel order,"[question about cancelling order, Order Number]",Extraction B is preferred for its better adher...
1,1,B,6,8,Extraction A correctly identifies the intent a...,"Extraction B provides a more specific intent, ...",inquire about cancelling order with specific o...,"[question about cancelling an order, specific ...",While both extractions capture the essence of ...
2,2,A,8,7,Extraction A accurately captures the intent to...,Extraction B provides a more detailed intent b...,cancel purchase,"[request to cancel order, providing order numb...",Extraction A is slightly better due to its cla...
3,3,A,2,2,Extraction A correctly identifies the intent t...,"Extraction B has the same issues as A, correct...",cancel purchase,[Order Number],Both extractions are lacking in symptom comple...
4,4,B,7,9,Extraction A accurately captures the intent to...,Extraction B correctly identifies the intent t...,cancel purchase,"[cannot afford the order, order number: {{Orde...",Extraction B is the better choice as it includ...


In [25]:
rewrite_prompt_template = """You are a prompt engineer. Improve the extraction prompt used for support-ticket triage.



Current prompt:

<current_prompt>

{{current_prompt}}

</current_prompt>



Sample outputs from pipeline (Call 1 and reviewed final output):

{{examples_json}}



Goal:

- Rewrite the prompt so that first-pass extraction is closer to reviewed final output.

- Keep output schema exactly: {\"intent\": string, \"symptoms\": string[]}

- Enforce: specific intent, complete symptoms, no hallucinations, atomic symptoms.

- Keep placeholder {{request}} in the prompt.



Return ONLY the improved prompt text (no markdown, no explanations).

"""



prompt_file = Path("support_ticket_triage_prompt.j2")

if not prompt_file.exists():

    raise FileNotFoundError("support_ticket_triage_prompt.j2 not found.")



if "results_df" not in globals() or results_df.empty:

    raise ValueError("results_df is missing or empty. Run Cell 3 first.")



current_prompt = prompt_file.read_text(encoding="utf-8")

examples = results_df[

    ["request", "call1_intent", "call1_symptoms", "final_intent", "final_symptoms"]

].to_dict(orient="records")

examples_json = json.dumps(examples, ensure_ascii=False, indent=2)



rewrite_instruction = Template(rewrite_prompt_template).render(

    current_prompt=current_prompt,

    examples_json=examples_json,

)



rewrite_response = client.chat.completions.create(

    model=model_name,

    messages=[

        {"role": "system", "content": "Return plain prompt text only."},

        {"role": "user", "content": rewrite_instruction},

    ],

    temperature=0,

)



improved_prompt = (rewrite_response.choices[0].message.content or "").strip()

if not improved_prompt:

    raise ValueError("OpenAI returned an empty prompt.")



if "{{request}}" not in improved_prompt:

    raise ValueError("Improved prompt is missing the required {{request}} placeholder.")



backup_file = prompt_file.with_suffix(".backup.j2")

backup_file.write_text(current_prompt + "\n", encoding="utf-8")



prompt_file.write_text(improved_prompt + "\n", encoding="utf-8")

print(f"Backup saved to: {backup_file}")

print("Updated support_ticket_triage_prompt.j2 with OpenAI-generated version.")


Backup saved to: support_ticket_triage_prompt.backup.j2
Updated support_ticket_triage_prompt.j2 with OpenAI-generated version.
